In [ ]:
# ── Memory setup: must run BEFORE any torch import ──────────────────────────
# expandable_segments reduces fragmentation (the primary cause of OOM on T4).
# This env var is also set inside run_experiment.py, but setting it here
# ensures it applies to ALL CUDA allocations in the notebook process too.
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Optional: limit per-process reserved memory so back-to-back model loads
# don't carry stale reservations into the next experiment.
import gc
def free_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

print('PYTORCH_ALLOC_CONF =', os.environ['PYTORCH_ALLOC_CONF'])
print('Memory helpers ready.')

# Salient-Quant: Complete Experiment Runner
**CS595-2 · Illinois Tech · Anshul Dani & Rohit Lahori**

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook runs all three milestones:
- **Exp 1** – GPT-2 small: baselines + ours (~15 min)
- **Exp 2** – GPT-2 medium: baselines + ours + 2 ablations, Milestone B (~60 min)
- **Exp 3** – LLaMA-3.2-1B: baselines + ours + 6 ablations + MMLU, Milestone C (~90 min)

## Step 0 — Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU (T4) and re-run.')

gpu = torch.cuda.get_device_properties(0)
print(f'GPU:  {gpu.name}')
print(f'VRAM: {gpu.total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')

## Step 1 — Clone Repository

In [ ]:
import os

GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'  # <-- change this
REPO_NAME = 'Simplified-Quantization-for-Edge-Ready-Language-Models'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git')
else:
    print('Repo already cloned, pulling latest...')
    os.system(f'cd {REPO_NAME} && git pull')

os.chdir(REPO_NAME)
print(f'Working directory: {os.getcwd()}')

In [ ]:
exec(open('/content/patch.py').read())


## Step 2 — Install Dependencies

In [ ]:
import subprocess
subprocess.run(['pip','install','-q','-r','requirements_colab.txt'])
print('Done.')

## Step 3 — HuggingFace Login (for LLaMA-3.2)

LLaMA-3.2 is gated. Request access at https://huggingface.co/meta-llama/Llama-3.2-1B
then get your token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # <-- paste your token
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in.')

## Step 4 — Mount Google Drive (recommended)

Preserves results if the session disconnects.

In [ ]:
SAVE_TO_DRIVE = True  # set False to skip

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/salient-quant-results'
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Results will sync to: {DRIVE_DIR}')
else:
    DRIVE_DIR = None
    print('Drive not mounted.')

## Helper — Pretty-print results

In [ ]:
import json
from pathlib import Path

def print_results(results_dir):
    p = Path(results_dir)
    files = list(p.glob('*_results.json'))
    if not files:
        print(f'No results in {results_dir}')
        return
    data = json.load(open(files[0]))
    print(f"\n{'='*65}")
    print(f"  {files[0].name}")
    print(f"{'='*65}")
    print(f"{'Method':<22} {'Bits':>6} {'WT2 PPL':>10} {'PTB PPL':>10} {'MMLU%':>8}")
    print('-'*60)
    for name, res in data.get('models', {}).items():
        bits  = res.get('avg_bits', '?')
        ppl2  = res.get('ppl_wikitext2', {}).get('perplexity', res.get('ppl_wikitext2', {}).get('error','?'))
        pplp  = res.get('ppl_ptb',      {}).get('perplexity', res.get('ppl_ptb',      {}).get('error','-'))
        mmlu  = res.get('mmlu',         {}).get('mmlu_macro_avg_pct', '-')
        fmt = lambda x: f'{x:.2f}' if isinstance(x, float) else str(x)
        print(f"{name:<22} {fmt(bits):>6} {fmt(ppl2):>10} {fmt(pplp):>10} {fmt(mmlu):>8}")

def print_ablation(results_dir, pattern='ablation_*.json'):
    for f in sorted(Path(results_dir).glob(pattern)):
        data = json.load(open(f))
        print(f"\n{'='*55}\n  {f.name}\n{'='*55}")
        print(f"{'Config':<28} {'PPL':>10} {'Bits':>8}")
        print('-'*48)
        for cfg_name, res in data.items():
            print(f"{cfg_name:<28} {res['perplexity']:>10.2f} {res['avg_bits']:>8.3f}")

print('Helpers ready.')

---
## Experiment 1 — GPT-2 Small Baseline + Ours
**~15 min on T4** | Config: `configs/gpt2_colab.yaml`

FP16 + Uniform INT2 + BitNet + Ours (1.61b), WikiText-2 perplexity.

In [ ]:
import os
os.system('python experiments/run_experiment.py --config configs/gpt2_colab.yaml')

In [ ]:
print_results('results/gpt2_colab')

from IPython.display import Image, display
import glob
for png in sorted(glob.glob('results/gpt2_colab/plots/*.png')):
    print(f'\n{png}')
    display(Image(png))

if SAVE_TO_DRIVE:
    os.system(f'cp -r results/gpt2_colab {DRIVE_DIR}/')

---
## Experiment 2 — GPT-2 Medium Full + Ablations (Milestone B)
**~60 min on T4** | Config: `configs/gpt2_full.yaml`

GPT-2 medium (345M): baselines + ours + ablations: salience_metric, granularity.
Evaluates WikiText-2, PTB, latency.

In [ ]:
os.system('python experiments/run_experiment.py --config configs/gpt2_full.yaml')

In [ ]:
print_results('results/gpt2_full')
print_ablation('results/gpt2_full')

for png in sorted(glob.glob('results/gpt2_full/plots/*.png')):
    print(f'\n{png}')
    display(Image(png))

if SAVE_TO_DRIVE:
    os.system(f'cp -r results/gpt2_full {DRIVE_DIR}/')

---
## Experiment 3 — LLaMA-3.2-1B Full + All Ablations + MMLU (Milestone C)
**~90 min on T4** | Config: `configs/llama_full.yaml`

LLaMA-3.2-1B: baselines + ours + 6 ablation studies + MMLU 5-shot.
Evaluates WikiText-2, PTB, MMLU, latency, memory.

> Requires HuggingFace token with LLaMA-3.2 access (Step 3 above).

In [ ]:
os.system('python experiments/run_experiment.py --config configs/llama_full.yaml')

In [ ]:
print_results('results/llama_full')
print_ablation('results/llama_full')

for png in sorted(glob.glob('results/llama_full/plots/*.png')):
    print(f'\n{png}')
    display(Image(png))

if SAVE_TO_DRIVE:
    os.system(f'cp -r results/llama_full {DRIVE_DIR}/')

---
## Summary — All Results

In [ ]:
print('\n' + '='*65)
print('CONSOLIDATED RESULTS')
print('='*65)
for label, d in [('GPT-2 small',   'results/gpt2_colab'),
                  ('GPT-2 medium', 'results/gpt2_full'),
                  ('LLaMA-3.2-1B', 'results/llama_full')]:
    print(f'\n  {label}')
    print_results(d)

if SAVE_TO_DRIVE:
    os.system(f'cp -r results {DRIVE_DIR}/')
    print(f'\nAll results synced to {DRIVE_DIR}')